In [3]:
import sklearn
import torchvision
import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
from torch.utils.data import DataLoader
from torchvision import datasets,transforms

In [4]:
transform=transforms.ToTensor()

In [5]:
train=datasets.MNIST(root="./data", train=True, download=True, transform=transform )
test=datasets.MNIST(root="./data", train=False, download=True, transform=transform )

In [6]:
train_loader=DataLoader(train, batch_size=64, shuffle=True)
test_loader=DataLoader(test, batch_size=64, shuffle=True)

In [7]:
class handwritting_reader(nn.Module):
    def __init__(self):
        super().__init__()
        self.fc1=nn.Linear(784,512)
        self.fc2=nn.Linear(512,256)
        self.fc3=nn.Linear(256,128)
        self.fc4=nn.Linear(128,10)
        self.dropout=nn.Dropout(0.3)
    
    def forward(self,x):
        x=x.view(-1,784)
        x=F.relu(self.fc1(x))
        x=self.dropout(x)
        x=F.relu(self.fc2(x))
        x=self.dropout(x)
        x=F.relu(self.fc3(x))
        x=self.fc4(x)

        return x

In [8]:
device= "cuda"if torch.cuda.is_available() else "cpu"

In [9]:
model= handwritting_reader().to(device)
loss_function= nn.CrossEntropyLoss()
optimizer= torch.optim.Adam(model.parameters(),lr=1e-4)

In [10]:
epochs=20
for epoch in range(epochs):
    model.train()
    total_loss=0
    for image,label in train_loader:
        image,label=image.to(device), label.to(device) 
        optimizer.zero_grad() 
        output= model(image) 
        loss=loss_function(output,label) 
        loss.backward() 
        optimizer.step()
        total_loss+=loss.item()
    print(f"Epoch {epoch+1}/{epochs} Loss: {total_loss/len(train_loader):.4f}")

Epoch 1/20 Loss: 0.6866
Epoch 2/20 Loss: 0.2853
Epoch 3/20 Loss: 0.2146
Epoch 4/20 Loss: 0.1722
Epoch 5/20 Loss: 0.1425
Epoch 6/20 Loss: 0.1213
Epoch 7/20 Loss: 0.1040
Epoch 8/20 Loss: 0.0941
Epoch 9/20 Loss: 0.0824
Epoch 10/20 Loss: 0.0715
Epoch 11/20 Loss: 0.0654
Epoch 12/20 Loss: 0.0577
Epoch 13/20 Loss: 0.0533
Epoch 14/20 Loss: 0.0473
Epoch 15/20 Loss: 0.0434
Epoch 16/20 Loss: 0.0392
Epoch 17/20 Loss: 0.0356
Epoch 18/20 Loss: 0.0336
Epoch 19/20 Loss: 0.0293
Epoch 20/20 Loss: 0.0290


In [12]:
model.eval()
correct=0
total=0
with torch.no_grad():
    for images,labels in test_loader:
        images,labels=images.to(device),labels.to(device)
        output=model(images)
        _,predicted=torch.max(output,1)
        total+=labels.size(0)
        correct+=(predicted==labels).sum().item()
    print(f"Test Accuracy: {100 * correct / total:.2f}%")

Test Accuracy: 98.04%
